# TT100K 两阶段路标识别

Kaggle 训练笔记本。检测用 YOLOv8，分类用手写 ResNet-18。两边共用同一份 `dataset.yaml` 和同一套类别编号。

运行环境请选择 **GPU T4 x2**。数据集添加 `braunge/tt100k`。全部写出文件都在 `/kaggle/working/`，输入目录只读。

按 Cell 1 到 Cell 10 的顺序运行。训完后下载 `best.pt`、`best_model.pth`、`dataset.yaml` 到本地 `weights/`，再用 `demo.py` 做推理。


## Cell 1 — 安装依赖


In [ ]:
!pip install -q ultralytics pyyaml matplotlib seaborn scikit-learn opencv-python-headless tqdm
print("依赖已安装。请按顺序运行后续单元。")


## Cell 2 — 检查环境并打印数据集目录树


In [ ]:
import json
import math
import os
import random
import re
import shutil
import sys
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

print("python", sys.version.split()[0])
print("cwd", os.getcwd())

try:
    import torch

    print("torch", torch.__version__)
    print("cuda_available", torch.cuda.is_available())
    print("gpu_count", torch.cuda.device_count())
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print(f"gpu[{index}]", props.name, f"{props.total_memory / 1024 ** 3:.1f} GB")
except ImportError:
    torch = None
    print("未安装 torch")

if os.environ.get("TT100K_WORKING"):
    WORKING = Path(os.environ["TT100K_WORKING"]).resolve()
elif Path("/kaggle/working").exists():
    WORKING = Path("/kaggle/working")
else:
    WORKING = Path("kaggle_working").resolve()
WORKING.mkdir(parents=True, exist_ok=True)

MIN_TRAIN_COUNT = 100
MIN_BOX = 16
SEED = 42
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

OFFICIAL_TYPES = (
    "i1,i10,i11,i12,i13,i14,i15,i2,i3,i4,i5,il100,il110,il50,il60,il70,il80,il90,io,ip,"
    "p1,p10,p11,p12,p13,p14,p15,p16,p17,p18,p19,p2,p20,p21,p22,p23,p24,p25,p26,p27,p28,"
    "p3,p4,p5,p6,p7,p8,p9,pa10,pa12,pa13,pa14,pa8,pb,pc,pg,ph1.5,ph2,ph2.1,ph2.2,ph2.4,"
    "ph2.5,ph2.8,ph2.9,ph3,ph3.2,ph3.5,ph3.8,ph4,ph4.2,ph4.3,ph4.5,ph4.8,ph5,ph5.3,ph5.5,"
    "pl10,pl100,pl110,pl120,pl15,pl20,pl25,pl30,pl35,pl40,pl5,pl50,pl60,pl65,pl70,pl80,pl90,"
    "pm10,pm13,pm15,pm1.5,pm2,pm20,pm25,pm30,pm35,pm40,pm46,pm5,pm50,pm55,pm8,pn,pne,po,"
    "pr10,pr100,pr20,pr30,pr40,pr45,pr50,pr60,pr70,pr80,ps,pw2,pw2.5,pw3,pw3.2,pw3.5,pw4,"
    "pw4.2,pw4.5,w1,w10,w12,w13,w16,w18,w20,w21,w22,w24,w28,w3,w30,w31,w32,w34,w35,w37,w38,"
    "w41,w42,w43,w44,w45,w46,w47,w48,w49,w5,w50,w55,w56,w57,w58,w59,w60,w62,w63,w66,w8,wo,"
    "i6,i7,i8,i9,ilx,p29,w29,w33,w36,w39,w4,w40,w51,w52,w53,w54,w6,w61,w64,w65,w67,w7,w9,"
    "pax,pd,pe,phx,plx,pmx,pnl,prx,pwx,w11,w14,w15,w17,w19,w2,w23,w25,w26,w27,pl0,pl4,pl3,"
    "pm2.5,ph4.4,pn40,ph3.3,ph2.6"
)
OFFICIAL_221 = OFFICIAL_TYPES.split(",")
if len(OFFICIAL_221) != 221 or len(set(OFFICIAL_221)) != 221:
    raise RuntimeError(f"官方类别表长度异常: {len(OFFICIAL_221)}")

MEANING_TABLE = {
    "i1": "步行",
    "i2": "非机动车行驶",
    "i3": "环岛行驶",
    "i4": "机动车行驶",
    "i5": "靠右侧道路行驶",
    "i6": "靠左侧道路行驶",
    "i7": "立体交叉直行和右转弯行驶",
    "i8": "立体交叉直行和左转弯行驶",
    "i9": "鸣喇叭",
    "i10": "向右转弯",
    "i11": "向左向右转弯",
    "i12": "向左转弯",
    "i13": "直行",
    "i14": "直行和向右转弯",
    "i15": "直行和向左转弯",
    "il50": "最低限速50",
    "il60": "最低限速60",
    "il70": "最低限速70",
    "il80": "最低限速80",
    "il90": "最低限速90",
    "il100": "最低限速100",
    "il110": "最低限速110",
    "io": "其他指示标志",
    "ip": "人行横道",
    "p1": "禁止超车",
    "p2": "禁止畜力车进入",
    "p3": "禁止大型客车驶入",
    "p4": "禁止电动三轮车驶入",
    "p5": "禁止掉头",
    "p6": "禁止非机动车进入",
    "p7": "禁止载货汽车左转",
    "p8": "禁止汽车拖、挂车驶入",
    "p9": "禁止行人进入",
    "p10": "禁止机动车驶入",
    "p11": "禁止鸣喇叭",
    "p12": "禁止二轮摩托车驶入",
    "p13": "禁止某两种车驶入",
    "p14": "禁止直行",
    "p15": "禁止人力车进入",
    "p16": "禁止人力货运三轮车进入",
    "p17": "禁止人力客运三轮车进入",
    "p18": "禁止三轮车机动车通行",
    "p19": "禁止向右转弯",
    "p20": "禁止向左向右转弯",
    "p21": "禁止直行和向右转弯",
    "p23": "禁止向左转弯",
    "p26": "禁止载货汽车驶入",
    "p27": "禁止运输危险物品车辆驶入",
    "p28": "禁止直行和向左转弯",
    "pb": "禁止通行",
    "pg": "减速让行",
    "pn": "禁止停车",
    "pne": "禁止驶入",
    "po": "其他禁止标志",
    "ph": "限高",
    "w": "警告标志",
    "pa": "限制轴重",
    "pm": "限重",
    "pr": "解除限速",
    "pbp": "禁止自行车和行人通行",
    "pr40": "解除限速40",
    "ps": "停车让行",
    "w13": "十字交叉",
    "w32": "施工",
    "w55": "注意儿童",
    "w57": "注意行人",
    "w59": "注意合流",
    "wo": "其他警告标志",
}


def meaning_of(code):
    code = str(code)
    if code in MEANING_TABLE:
        return MEANING_TABLE[code]
    matched = re.fullmatch(r"pl(\d+(?:\.\d+)?)", code)
    if matched:
        return f"限速{matched.group(1)}"
    matched = re.fullmatch(r"il(\d+(?:\.\d+)?)", code)
    if matched:
        return f"最低限速{matched.group(1)}"
    matched = re.fullmatch(r"ph(\d+(?:\.\d+)?)", code)
    if matched:
        return f"限高{matched.group(1)}米"
    matched = re.fullmatch(r"pm(\d+(?:\.\d+)?)", code)
    if matched:
        return f"限重{matched.group(1)}吨"
    matched = re.fullmatch(r"pw(\d+(?:\.\d+)?)", code)
    if matched:
        return f"限宽{matched.group(1)}米"
    matched = re.fullmatch(r"pr(\d+(?:\.\d+)?)", code)
    if matched:
        return f"解除限速{matched.group(1)}"
    matched = re.fullmatch(r"pa(\d+(?:\.\d+)?)", code)
    if matched:
        return f"限制轴重{matched.group(1)}吨"
    return code


def score_dataset_dir(path):
    try:
        children = list(path.iterdir())
    except OSError:
        return 0
    names = {child.name.lower() for child in children}
    score = 0
    if "annotations.json" in names or (path / "data" / "annotations.json").exists():
        score += 100
    if any(name.endswith((".yaml", ".yml")) for name in names):
        score += 40
    if "images" in names and "labels" in names:
        score += 80
    if "train" in names:
        score += 20
    if "test" in names:
        score += 10
    return score


def locate_input():
    env_path = os.environ.get("TT100K_INPUT")
    if env_path:
        path = Path(env_path)
        if not path.exists():
            raise FileNotFoundError(f"TT100K_INPUT 不存在: {path}")
        return path.resolve()

    preferred = [
        Path("/kaggle/input/braunge/tt100k"),
        Path("/kaggle/input/tt100k"),
        Path("/kaggle/input/datasets/braunge/tt100k"),
    ]
    for path in preferred:
        if path.exists():
            return path.resolve()

    base = Path("/kaggle/input")
    if not base.exists():
        raise FileNotFoundError(
            "未找到数据集。请添加 braunge/tt100k，并确认挂载在 /kaggle/input/braunge/tt100k。"
        )

    candidates = []
    for dirpath, dirnames, filenames in os.walk(base):
        current = Path(dirpath)
        depth = len(current.relative_to(base).parts)
        if depth > 3 or len(filenames) > 400:
            dirnames[:] = []
        dirnames[:] = [name for name in dirnames if not name.startswith(".")]
        score = score_dataset_dir(current)
        if score > 0:
            candidates.append((score, -len(current.parts), current))
    if not candidates:
        raise FileNotFoundError(f"在 {base} 下没有找到 TT100K 数据目录")
    candidates.sort(reverse=True)
    return candidates[0][2].resolve()


def print_tree(root, max_depth=3, max_entries=25):
    print(f"数据集根目录: {root}")

    def walk(path, depth):
        if depth > max_depth:
            return
        try:
            entries = sorted(path.iterdir(), key=lambda item: (not item.is_dir(), item.name.lower()))
        except OSError as exc:
            print("  " * depth + f"[无法读取] {exc}")
            return
        directories = [entry for entry in entries if entry.is_dir()]
        files = [entry for entry in entries if entry.is_file()]
        for directory in directories[:max_entries]:
            child_dirs = 0
            child_files = 0
            try:
                for child in directory.iterdir():
                    if child.is_dir():
                        child_dirs += 1
                    else:
                        child_files += 1
            except OSError:
                pass
            print("  " * depth + f"{directory.name}/  (子目录 {child_dirs}, 文件 {child_files})")
            if child_files > 40 and depth >= 1:
                continue
            walk(directory, depth + 1)
        if len(directories) > max_entries:
            print("  " * depth + f"... 另有 {len(directories) - max_entries} 个子目录")
        for file_path in files[:max_entries]:
            size_kb = file_path.stat().st_size / 1024
            print("  " * depth + f"{file_path.name}  ({size_kb:.1f} KB)")
        if len(files) > max_entries:
            print("  " * depth + f"... 另有 {len(files) - max_entries} 个文件")

    walk(root, 0)


INPUT_ROOT = locate_input()
print("输入目录:", INPUT_ROOT)
print("输出目录:", WORKING)
print_tree(INPUT_ROOT)


## Cell 3 — 数据探索

统计每个划分里的类别实例，并画出训练集分布。实例数达到 100 的类别用深色标出。


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SKIP_TEXT = {"classes.txt", "ids.txt", "readme.txt", "license.txt"}


def split_of(path, root):
    parts = [part.lower() for part in path.relative_to(root).parts[:-1]]
    for part in parts:
        if part in {"train", "training"}:
            return "train"
        if part in {"val", "valid", "validation"}:
            return "val"
        if part == "test":
            return "test"
        if part == "other":
            return "val"
    return "train"


def clean_yaml_scalar(text):
    text = text.split(" #", 1)[0].strip()
    if len(text) >= 2 and text[0] == text[-1] and text[0] in {"'", '"'}:
        return text[1:-1]
    return text


def parse_names_block(text):
    bracket = re.search(r"names\s*:\s*\[(.*?)\]", text, flags=re.S)
    if bracket:
        quoted = re.findall(r"['\"]([^'\"]+)['\"]", bracket.group(1))
        if quoted:
            return quoted
    lines = text.splitlines()
    start = None
    for index, line in enumerate(lines):
        if re.match(r"^names\s*:", line):
            start = index
            break
    if start is None:
        return None
    inline = re.match(r"^names\s*:\s*\[(.*)\]\s*$", lines[start])
    if inline:
        return [clean_yaml_scalar(part) for part in inline.group(1).split(",") if part.strip()]

    mapping = {}
    sequence = []
    mode = None
    for line in lines[start + 1 :]:
        if not line.strip() or line.lstrip().startswith("#"):
            continue
        if not line.startswith((" ", "\t")):
            break
        item = re.match(r"^\s*-\s*(.+?)\s*$", line)
        pair = re.match(r"^\s*([^:#]+)\s*:\s*(.+?)\s*$", line)
        if item and mode != "map":
            mode = "list"
            sequence.append(clean_yaml_scalar(item.group(1)))
        elif pair and mode != "list":
            mode = "map"
            mapping[int(float(clean_yaml_scalar(pair.group(1))))] = clean_yaml_scalar(pair.group(2))
        else:
            break
    if mapping:
        size = max(mapping) + 1
        return [mapping.get(index, f"class_{index}") for index in range(size)]
    if sequence:
        return sequence
    return None


def collect_paths(root):
    images = []
    texts = []
    annotations = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [name for name in dirnames if not name.startswith(".")]
        current = Path(dirpath)
        if len(current.relative_to(root).parts) > 6:
            dirnames[:] = []
        for filename in filenames:
            path = current / filename
            suffix = path.suffix.lower()
            lowered = filename.lower()
            if suffix in IMAGE_EXTS:
                images.append(path)
            elif lowered in {"annotations.json", "annotations_all.json"}:
                annotations.append(path)
            elif suffix == ".txt" and lowered not in SKIP_TEXT:
                texts.append(path)
    annotations.sort(key=lambda path: (path.name != "annotations.json", len(path.parts)))
    return images, texts, annotations


def looks_like_yolo_label(path):
    try:
        if path.stat().st_size == 0:
            return True
        text = path.read_text(encoding="utf-8", errors="ignore")
    except OSError:
        return False
    checked = 0
    for line in text.splitlines():
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) < 5:
            return False
        try:
            int(float(parts[0]))
            values = [float(item) for item in parts[1:5]]
        except ValueError:
            return False
        if not all(-0.05 <= value <= 1.5 for value in values):
            return False
        checked += 1
        if checked >= 3:
            break
    return checked > 0


def load_name_list(root):
    yaml_files = []
    class_files = []
    for dirpath, dirnames, filenames in os.walk(root):
        current = Path(dirpath)
        if len(current.relative_to(root).parts) > 4:
            dirnames[:] = []
            continue
        dirnames[:] = [name for name in dirnames if not name.startswith(".")]
        for filename in filenames:
            path = current / filename
            if filename.lower().endswith((".yaml", ".yml")):
                yaml_files.append(path)
            elif filename.lower() == "classes.txt":
                class_files.append(path)

    best_names = None
    best_path = None
    for path in yaml_files:
        try:
            names = parse_names_block(path.read_text(encoding="utf-8", errors="ignore"))
        except OSError:
            continue
        if names and (best_names is None or len(names) > len(best_names) or (len(names) == len(best_names) and 'yolov8' in path.name.lower() and 'yolov8' not in str(best_path).lower())):
            best_names = names
            best_path = path
    if best_names:
        print(f"类别名来自 {best_path}，共 {len(best_names)} 类")
        return best_names

    for path in class_files:
        lines = path.read_text(encoding="utf-8", errors="ignore").splitlines()
        names = [clean_yaml_scalar(line) for line in lines if line.strip() and not line.strip().startswith("#")]
        if names:
            print(f"类别名来自 {path}，共 {len(names)} 类")
            return names
    print("没有找到 data.yaml 或 classes.txt")
    return None


def parse_yolo_file(path):
    objects = []
    text = path.read_text(encoding="utf-8", errors="ignore")
    for line in text.splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        try:
            class_id = int(float(parts[0]))
            cx, cy, width, height = [float(item) for item in parts[1:5]]
        except ValueError:
            continue
        objects.append((class_id, cx, cy, width, height))
    return objects


def clip_xywhn(cx, cy, width, height):
    x1 = max(0.0, cx - width / 2.0)
    y1 = max(0.0, cy - height / 2.0)
    x2 = min(1.0, cx + width / 2.0)
    y2 = min(1.0, cy + height / 2.0)
    box_w = x2 - x1
    box_h = y2 - y1
    if box_w <= 0 or box_h <= 0:
        return None
    return (x1 + x2) / 2.0, (y1 + y2) / 2.0, box_w, box_h


def name_for(orig_names, class_id):
    if 0 <= class_id < len(orig_names):
        return orig_names[class_id]
    return f"class_{class_id}"


def records_from_yolo(root, images, label_files, orig_names):
    image_index = {}
    stem_count = Counter(path.stem for path in images)
    unique_stem = {path.stem: path for path in images if stem_count[path.stem] == 1}
    for image_path in images:
        image_index[(split_of(image_path, root), image_path.stem)] = image_path

    records = []
    missing_images = 0
    for label_path in tqdm(label_files, desc="读取 YOLO 标签"):
        objects = parse_yolo_file(label_path)
        if not objects:
            continue
        split = split_of(label_path, root)
        image_path = image_index.get((split, label_path.stem)) or unique_stem.get(label_path.stem)
        if image_path is None:
            missing_images += 1
            continue
        packed = []
        for class_id, cx, cy, width, height in objects:
            xywhn = clip_xywhn(cx, cy, width, height)
            if xywhn is None:
                continue
            packed.append(
                {
                    "orig_id": class_id,
                    "name": name_for(orig_names, class_id),
                    "xywhn": xywhn,
                }
            )
        if packed:
            records.append({"split": split, "image": image_path, "objects": packed})
    print(f"YOLO 记录 {len(records)} 张图，缺图标签 {missing_images}")
    return records


def split_from_rel(relpath):
    text = str(relpath).replace("\\", "/").lower()
    parts = text.split("/")
    if "train" in parts or text.startswith("train"):
        return "train"
    if "test" in parts:
        return "test"
    return "val"


def resolve_anno_image(root, relpath, by_name):
    relative = Path(str(relpath).replace("\\", "/"))
    candidates = [
        root / relative,
        root / "data" / relative,
        root / relative.name,
    ]
    for folder in ("train", "test", "other", "val", "valid"):
        candidates.append(root / folder / relative.name)
        candidates.append(root / "data" / folder / relative.name)
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    return by_name.get(relative.name)


def xyxy_to_xywhn(xmin, ymin, xmax, ymax, width, height):
    xmin = max(0.0, min(float(width), float(xmin)))
    ymin = max(0.0, min(float(height), float(ymin)))
    xmax = max(0.0, min(float(width), float(xmax)))
    ymax = max(0.0, min(float(height), float(ymax)))
    box_w = xmax - xmin
    box_h = ymax - ymin
    if box_w < 1 or box_h < 1:
        return None
    return (
        ((xmin + xmax) / 2.0) / width,
        ((ymin + ymax) / 2.0) / height,
        box_w / width,
        box_h / height,
    )


def records_from_annotations(root, images, anno_path):
    payload = json.loads(anno_path.read_text(encoding="utf-8"))
    images_info = payload.get("imgs") or payload.get("imgs".upper())
    if not isinstance(images_info, dict):
        raise RuntimeError(f"{anno_path} 缺少 imgs")
    by_name = {}
    for image_path in images:
        by_name.setdefault(image_path.name, image_path)

    orig_names = list(OFFICIAL_221)
    name_to_id = {name: index for index, name in enumerate(orig_names)}
    records = []
    missing_images = 0
    for item in tqdm(images_info.values(), total=len(images_info), desc="读取 annotations"):
        relpath = item.get("path") or ""
        image_path = resolve_anno_image(root, relpath, by_name)
        if image_path is None:
            missing_images += 1
            continue
        try:
            with Image.open(image_path) as raw:
                width, height = raw.size
        except OSError:
            missing_images += 1
            continue
        packed = []
        for obj in item.get("objects") or []:
            category = str(obj.get("category", "")).strip()
            bbox = obj.get("bbox") or {}
            if not category or not all(key in bbox for key in ("xmin", "ymin", "xmax", "ymax")):
                continue
            if category not in name_to_id:
                name_to_id[category] = len(orig_names)
                orig_names.append(category)
            xywhn = xyxy_to_xywhn(bbox["xmin"], bbox["ymin"], bbox["xmax"], bbox["ymax"], width, height)
            if xywhn is None:
                continue
            packed.append({"orig_id": name_to_id[category], "name": category, "xywhn": xywhn})
        if packed:
            records.append({"split": split_from_rel(relpath), "image": image_path, "objects": packed})
    print(f"标注记录 {len(records)} 张图，缺图 {missing_images}，类别表 {len(orig_names)}")
    return records, orig_names


def ensure_name_table(orig_names, records):
    max_id = max(obj["orig_id"] for rec in records for obj in rec["objects"])
    if orig_names is None:
        if max_id < len(OFFICIAL_221):
            print("警告: 标签只有数字 ID，回退到清华教程 annos['types'] 的 221 类顺序。")
            orig_names = list(OFFICIAL_221)
        else:
            raise RuntimeError("标签只有数字 ID，且超出官方 221 类范围。请提供带 names 的 yaml。")
    if max_id >= len(orig_names):
        print(f"警告: 最大类别 ID {max_id} 超出名单长度 {len(orig_names)}，缺口用 class_编号 补齐。")
        orig_names = list(orig_names) + [f"class_{index}" for index in range(len(orig_names), max_id + 1)]
    else:
        orig_names = list(orig_names)
    for rec in records:
        for obj in rec["objects"]:
            obj["name"] = orig_names[obj["orig_id"]]
    return orig_names


def build_records(root):
    images, texts, annotations = collect_paths(root)
    yolo_labels = [path for path in texts if looks_like_yolo_label(path)]
    print(f"图片 {len(images)}，疑似 YOLO 标签 {len(yolo_labels)}，标注文件 {len(annotations)}")
    if len(yolo_labels) >= 50:
        orig_names = load_name_list(root)
        records = records_from_yolo(root, images, yolo_labels, orig_names or [])
        if not records:
            raise RuntimeError("找到了 YOLO 标签，但没有配上图片")
        orig_names = ensure_name_table(orig_names, records)
        return records, orig_names, "yolo"
    if annotations:
        print(f"使用标注文件 {annotations[0]}")
        records, orig_names = records_from_annotations(root, images, annotations[0])
        if not records:
            raise RuntimeError("annotations.json 没有匹配到图片")
        return records, orig_names, "annotations"
    if yolo_labels:
        orig_names = load_name_list(root)
        records = records_from_yolo(root, images, yolo_labels, orig_names or [])
        if not records:
            raise RuntimeError("YOLO 标签没有配上图片")
        orig_names = ensure_name_table(orig_names, records)
        return records, orig_names, "yolo"
    raise RuntimeError(f"{root} 里既没有 YOLO 标签，也没有 annotations.json")


RAW_RECORDS, ORIG_NAMES, DATA_SOURCE = build_records(INPUT_ROOT)
print("数据来源:", DATA_SOURCE)
print("原始类别数:", len(ORIG_NAMES))

split_counts = Counter(rec["split"] for rec in RAW_RECORDS)
object_counts = Counter()
train_counts = Counter()
for rec in RAW_RECORDS:
    for obj in rec["objects"]:
        object_counts[rec["split"]] += 1
        if rec["split"] == "train":
            train_counts[obj["orig_id"]] += 1
print("有标注的图片:", dict(split_counts))
print("标注实例:", dict(object_counts))
kept_preview = sum(1 for count in train_counts.values() if count >= MIN_TRAIN_COUNT)
print(f"训练集实例数 >= {MIN_TRAIN_COUNT} 的类别: {kept_preview}")

ranked = sorted(train_counts.items(), key=lambda item: item[1], reverse=True)
labels = [ORIG_NAMES[class_id] if class_id < len(ORIG_NAMES) else str(class_id) for class_id, _ in ranked]
values = [count for _, count in ranked]
colors = ["#1d4e89" if count >= MIN_TRAIN_COUNT else "#c5cdd6" for count in values]
figure_width = max(14, len(labels) * 0.28)
plt.figure(figsize=(figure_width, 6))
plt.bar(range(len(values)), values, color=colors)
plt.axhline(MIN_TRAIN_COUNT, color="#b23a48", linestyle="--", linewidth=1)
plt.xticks(range(len(labels)), labels, rotation=90, fontsize=7)
plt.ylabel("train instances")
plt.title(f"TT100K train class counts (highlight >= {MIN_TRAIN_COUNT})")
plt.tight_layout()
chart_path = WORKING / "class_distribution_train.png"
plt.savefig(chart_path, dpi=120)
plt.show()
print("柱状图:", chart_path)


## Cell 4 — 保留训练集实例数不少于 100 的类别，并重映射 ID

新编号按原 ID 从小到大变成 `0 .. n-1`。图片用符号链接指回只读输入，标签写到 `/kaggle/working/tt100k45/`。


In [ ]:
train_counts = Counter()
for rec in RAW_RECORDS:
    if rec["split"] != "train":
        continue
    for obj in rec["objects"]:
        train_counts[obj["orig_id"]] += 1

kept_old_ids = sorted(class_id for class_id, count in train_counts.items() if count >= MIN_TRAIN_COUNT)
if not kept_old_ids:
    raise RuntimeError(f"没有任何类别的训练实例数达到 {MIN_TRAIN_COUNT}")

OLD_TO_NEW = {old_id: new_id for new_id, old_id in enumerate(kept_old_ids)}
KEPT_NAMES = []
for old_id in kept_old_ids:
    name = ORIG_NAMES[old_id] if old_id < len(ORIG_NAMES) else f"class_{old_id}"
    KEPT_NAMES.append(str(name))
if len(set(KEPT_NAMES)) != len(KEPT_NAMES):
    KEPT_NAMES = [f"{name}__{old_id}" for name, old_id in zip(KEPT_NAMES, kept_old_ids)]
KEPT_MEANINGS = [meaning_of(name.split("__", 1)[0]) for name in KEPT_NAMES]

print(f"保留 {len(KEPT_NAMES)} 类。官方论文子集一般是 45 类；若不是 45，仍按 >= {MIN_TRAIN_COUNT} 继续。")
print(f"{'new_id':>6}  {'old_id':>6}  {'train':>7}  code  meaning")
for new_id, old_id in enumerate(kept_old_ids):
    print(f"{new_id:6d}  {old_id:6d}  {train_counts[old_id]:7d}  {KEPT_NAMES[new_id]}  {KEPT_MEANINGS[new_id]}")

DET_ROOT = WORKING / "tt100k45"
if DET_ROOT.exists():
    shutil.rmtree(DET_ROOT)
LINK_MODE = {"mode": None}


def link_or_copy(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_symlink() or dst.exists():
        dst.unlink()
    if LINK_MODE["mode"] == "copy":
        shutil.copy2(src, dst)
        return
    try:
        os.symlink(os.fspath(Path(src).resolve()), os.fspath(dst))
        LINK_MODE["mode"] = "symlink"
    except OSError:
        shutil.copy2(src, dst)
        LINK_MODE["mode"] = "copy"
        print("当前环境不能创建符号链接，图片改为复制。")


def yaml_quote(value):
    text = str(value).replace("\\", "\\\\").replace('"', '\\"')
    return f'"{text}"'


split_image_counts = Counter()
split_object_counts = Counter()
for rec in RAW_RECORDS:
    lines = []
    for obj in rec["objects"]:
        new_id = OLD_TO_NEW.get(obj["orig_id"])
        if new_id is None:
            continue
        cx, cy, width, height = obj["xywhn"]
        lines.append(f"{new_id} {cx:.6f} {cy:.6f} {width:.6f} {height:.6f}\n")
    if not lines:
        continue
    split = rec["split"]
    image_path = Path(rec["image"])
    destination = DET_ROOT / "images" / split / image_path.name
    link_or_copy(image_path, destination)
    label_path = DET_ROOT / "labels" / split / f"{image_path.stem}.txt"
    label_path.parent.mkdir(parents=True, exist_ok=True)
    label_path.write_text("".join(lines), encoding="utf-8", newline="\n")
    split_image_counts[split] += 1
    split_object_counts[split] += len(lines)

if split_image_counts["train"] == 0:
    raise RuntimeError("筛选后训练集为空")
print("筛选后图片数:", dict(split_image_counts))
print("筛选后实例数:", dict(split_object_counts))
print("图片链接方式:", LINK_MODE["mode"])

present = [split for split in ("train", "val", "test") if split_image_counts[split] > 0]
if "val" in present:
    val_rel = "images/val"
elif "test" in present:
    val_rel = "images/test"
    print("没有独立验证集，YOLO 的 val 指向 images/test。")
else:
    val_rel = "images/train"
    print("警告: 没有验证集或测试集，YOLO 的 val 暂时指向 images/train。")

yaml_lines = [
    f"path: {DET_ROOT.resolve().as_posix()}",
    "train: images/train",
    f"val: {val_rel}",
]
if "test" in present:
    yaml_lines.append("test: images/test")
yaml_lines.append(f"nc: {len(KEPT_NAMES)}")
yaml_lines.append("names:")
yaml_lines.extend(f"  - {yaml_quote(name)}" for name in KEPT_NAMES)
yaml_lines.append("meanings:")
yaml_lines.extend(f"  - {yaml_quote(meaning)}" for meaning in KEPT_MEANINGS)
yaml_text = "\n".join(yaml_lines) + "\n"

DATASET_YAML = DET_ROOT / "dataset.yaml"
DATASET_YAML.write_text(yaml_text, encoding="utf-8", newline="\n")
EXPORT_YAML = WORKING / "dataset.yaml"
EXPORT_YAML.write_text(yaml_text, encoding="utf-8", newline="\n")
print("检测数据:", DET_ROOT)
print("下载用名单:", EXPORT_YAML)


## Cell 5 — 按 Ground Truth 裁剪路标，构建 ResNet 分类集

小图保存为 224×224。短边小于 16 像素的框丢弃。全部有效裁剪按类别做 8:1:1 划分，不使用会把左右转标志翻过去的水平翻转。


In [ ]:
def pixel_box(xywhn, width, height, min_side):
    cx, cy, box_w, box_h = xywhn
    x1 = int(math.floor((cx - box_w / 2.0) * width))
    y1 = int(math.floor((cy - box_h / 2.0) * height))
    x2 = int(math.ceil((cx + box_w / 2.0) * width))
    y2 = int(math.ceil((cy + box_h / 2.0) * height))
    x1 = max(0, min(width - 1, x1))
    y1 = max(0, min(height - 1, y1))
    x2 = max(0, min(width, x2))
    y2 = max(0, min(height, y2))
    if x2 <= x1 or y2 <= y1:
        return None
    if min(x2 - x1, y2 - y1) < min_side:
        return None
    return x1, y1, x2, y2


candidates = []
for rec in RAW_RECORDS:
    for obj in rec["objects"]:
        new_id = OLD_TO_NEW.get(obj["orig_id"])
        if new_id is None:
            continue
        candidates.append({"image": Path(rec["image"]), "new_id": new_id, "xywhn": obj["xywhn"]})

grouped = defaultdict(list)
for item in candidates:
    grouped[item["image"]].append(item)

valid = []
unreadable = 0
too_small = 0
for image_path, items in tqdm(grouped.items(), desc="检查路标尺寸"):
    try:
        with Image.open(image_path) as raw:
            width, height = raw.size
    except OSError:
        unreadable += 1
        too_small += len(items)
        continue
    for item in items:
        box = pixel_box(item["xywhn"], width, height, MIN_BOX)
        if box is None:
            too_small += 1
            continue
        valid.append({**item, "box": box})

print(f"候选框 {len(candidates)}，有效裁剪 {len(valid)}，过小或越界 {too_small}，坏图组 {unreadable}")
if not valid:
    raise RuntimeError("没有可用于分类的路标裁剪")


def split_one_class(indices, rng):
    rng.shuffle(indices)
    count = len(indices)
    if count == 1:
        return indices, [], []
    if count == 2:
        return indices[:1], indices[1:], []
    n_val = count // 10
    n_test = count // 10
    if n_val == 0:
        n_val = 1
    if n_test == 0:
        n_test = 1
    if n_val + n_test >= count:
        n_val = 1
        n_test = 1
    n_train = count - n_val - n_test
    return indices[:n_train], indices[n_train : n_train + n_val], indices[n_train + n_val :]


rng = random.Random(SEED)
buckets = defaultdict(list)
for index, item in enumerate(valid):
    buckets[item["new_id"]].append(index)
assigned = {}
for class_id, indices in buckets.items():
    train_idx, val_idx, test_idx = split_one_class(indices, rng)
    for index in train_idx:
        assigned[index] = "train"
    for index in val_idx:
        assigned[index] = "val"
    for index in test_idx:
        assigned[index] = "test"

for class_id, indices in sorted(buckets.items()):
    counts = Counter(assigned[index] for index in indices)
    print(
        f"class {class_id:02d} {KEPT_NAMES[class_id]:8} "
        f"train {counts['train']:4d}  val {counts['val']:4d}  test {counts['test']:4d}"
    )

CLS_ROOT = WORKING / "cls_dataset"
if CLS_ROOT.exists():
    shutil.rmtree(CLS_ROOT)
for split in ("train", "val", "test"):
    for class_id in range(len(KEPT_NAMES)):
        (CLS_ROOT / split / f"{class_id:03d}").mkdir(parents=True, exist_ok=True)

resample = getattr(getattr(Image, "Resampling", Image), "BICUBIC")
manifests = {split: [] for split in ("train", "val", "test")}
save_groups = defaultdict(list)
for index, item in enumerate(valid):
    save_groups[item["image"]].append((index, item))

saved = 0
for image_path, items in tqdm(save_groups.items(), desc="裁剪并保存"):
    with Image.open(image_path) as raw:
        image = raw.convert("RGB").copy()
    for index, item in items:
        split = assigned[index]
        filename = f"{image_path.stem}_{index:06d}.jpg"
        destination = CLS_ROOT / split / f"{item['new_id']:03d}" / filename
        image.crop(item["box"]).resize((224, 224), resample).save(destination, quality=95)
        manifests[split].append(f"{destination.resolve().as_posix()}\t{item['new_id']}\n")
        saved += 1

for split, lines in manifests.items():
    (CLS_ROOT / f"{split}.txt").write_text("".join(lines), encoding="utf-8", newline="\n")
    print(split, len(lines))
total = sum(len(lines) for lines in manifests.values())
train_per_class = Counter()
for line in manifests["train"]:
    train_per_class[int(line.rsplit("\t", 1)[1])] += 1
missing = [KEPT_NAMES[index] for index in range(len(KEPT_NAMES)) if train_per_class[index] == 0]
if missing:
    print("警告: 下列类别没有足够大的训练裁剪，编号仍然保留:", ", ".join(missing))
print(f"分类集合计 {total}，比例 train:val:test = {len(manifests['train'])}:{len(manifests['val'])}:{len(manifests['test'])}")
print("已保存", saved, "张到", CLS_ROOT)
if not manifests["train"]:
    raise RuntimeError("分类训练集为空")


## Cell 6 — 手写 ResNet-18

`BasicBlock` 含两层 3×3 卷积。步长或通道变化时，捷径用 1×1 卷积加 BatchNorm 对齐维度。结构必须与本地 `demo.py` 一致，否则权重对不上。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class BasicBlock(nn.Module):
    """ResNet 基础残差块。通道或步长变化时，用 1x1 卷积把捷径分支对齐到主分支。"""

    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes * self.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * self.expansion),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out)


class ResNet18(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, num_blocks=2, stride=1)
        self.layer2 = self._make_layer(128, num_blocks=2, stride=2)
        self.layer3 = self._make_layer(256, num_blocks=2, stride=2)
        self.layer4 = self._make_layer(512, num_blocks=2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * BasicBlock.expansion, num_classes)

    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for block_stride in strides:
            layers.append(BasicBlock(self.in_planes, planes, stride=block_stride))
            self.in_planes = planes * BasicBlock.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.maxpool(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        return self.fc(out)


_probe = ResNet18(len(KEPT_NAMES))
print("ResNet-18 参数量:", sum(parameter.numel() for parameter in _probe.parameters()))
print("输出类别数:", len(KEPT_NAMES))
del _probe


## Cell 7 — 训练 ResNet-18

损失为 CrossEntropyLoss，优化器为 AdamW，学习率用 CosineAnnealingLR。多张 GPU 时用 `nn.DataParallel`，并开启 AMP。`num_workers=0`。每 5 轮保存一次 checkpoint，验证集最优另存为 `best_model.pth`。


In [ ]:
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

EPOCHS = 20
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0


class SignDataset(Dataset):
    def __init__(self, manifest, transform):
        self.items = []
        with open(manifest, encoding="utf-8") as handle:
            for line in handle:
                line = line.strip()
                if not line:
                    continue
                path, class_id = line.rsplit("\t", 1)
                self.items.append((path, int(class_id)))
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        path, class_id = self.items[index]
        with Image.open(path) as raw:
            image = raw.convert("RGB").copy()
        return self.transform(image), class_id


train_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)
eval_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)


def pick_manifest(name):
    path = CLS_ROOT / f"{name}.txt"
    if path.exists() and path.stat().st_size > 0:
        return path
    return None


train_manifest = pick_manifest("train")
val_manifest = pick_manifest("val") or pick_manifest("test") or train_manifest
if train_manifest is None:
    raise RuntimeError("缺少分类训练集清单")
if val_manifest != CLS_ROOT / "val.txt":
    print("验证清单改用:", val_manifest)

train_loader = DataLoader(
    SignDataset(train_manifest, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    SignDataset(val_manifest, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
print("train batches", len(train_loader), "val batches", len(val_loader))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNet18(num_classes=len(KEPT_NAMES))
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print("ResNet 使用 DataParallel, GPU 数:", torch.cuda.device_count())
else:
    print("ResNet 使用单设备:", device)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
use_amp = device.type == "cuda"


def amp_context():
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast(device_type=device.type, enabled=use_amp)
    return torch.cuda.amp.autocast(enabled=use_amp)


if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
    try:
        scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    except TypeError:
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
else:
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


def unwrap_state_dict(module):
    raw = module.module if isinstance(module, nn.DataParallel) else module
    return raw.state_dict()


def save_checkpoint(path, module, epoch, val_acc):
    payload = {
        "state_dict": unwrap_state_dict(module),
        "class_names": list(KEPT_NAMES),
        "meanings": list(KEPT_MEANINGS),
        "num_classes": len(KEPT_NAMES),
        "epoch": epoch,
        "val_acc": float(val_acc),
        "normalize": {"mean": list(IMAGENET_MEAN), "std": list(IMAGENET_STD)},
    }
    torch.save(payload, path)


def evaluate(module, loader):
    module.eval()
    correct = 0
    total = 0
    loss_sum = 0.0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with amp_context():
                logits = module(images)
                loss = criterion(logits, labels)
            prediction = logits.argmax(dim=1)
            correct += (prediction == labels).sum().item()
            total += labels.size(0)
            loss_sum += loss.item() * labels.size(0)
    return loss_sum / max(total, 1), correct / max(total, 1)


CKPT_DIR = WORKING / "resnet_checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = WORKING / "best_model.pth"
best_acc = -1.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0
    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with amp_context():
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        prediction = logits.argmax(dim=1)
        running_loss += loss.item() * labels.size(0)
        running_correct += (prediction == labels).sum().item()
        running_total += labels.size(0)
    learning_rate = optimizer.param_groups[0]["lr"]
    scheduler.step()
    train_loss = running_loss / max(running_total, 1)
    train_acc = running_correct / max(running_total, 1)
    val_loss, val_acc = evaluate(model, val_loader)
    print(
        f"epoch {epoch:02d}/{EPOCHS}  lr {learning_rate:.6f}  "
        f"train_loss {train_loss:.4f}  train_acc {train_acc:.4f}  "
        f"val_loss {val_loss:.4f}  val_acc {val_acc:.4f}"
    )
    if val_acc > best_acc:
        best_acc = val_acc
        save_checkpoint(BEST_MODEL_PATH, model, epoch, val_acc)
        print("保存最优权重:", BEST_MODEL_PATH, f"val_acc {best_acc:.4f}")
    if epoch % 5 == 0:
        epoch_path = CKPT_DIR / f"epoch_{epoch:03d}.pth"
        save_checkpoint(epoch_path, model, epoch, val_acc)
        print("周期检查点:", epoch_path)

print("ResNet 训练结束，最优验证准确率:", round(best_acc, 4))


## Cell 8 — 评估 ResNet-18

在分类测试集上输出混淆矩阵和 classification report。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

test_manifest = CLS_ROOT / "test.txt"
if not test_manifest.exists() or test_manifest.stat().st_size == 0:
    raise RuntimeError("分类测试集为空，无法评估")

test_loader = DataLoader(
    SignDataset(test_manifest, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)


def load_resnet(path):
    try:
        checkpoint = torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        checkpoint = torch.load(path, map_location=device)
    cleaned = {}
    for key, value in checkpoint["state_dict"].items():
        name = key
        while name.startswith("module."):
            name = name[len("module.") :]
        cleaned[name] = value
    network = ResNet18(checkpoint["num_classes"]).to(device)
    network.load_state_dict(cleaned)
    network.eval()
    return network, checkpoint


eval_model, eval_checkpoint = load_resnet(BEST_MODEL_PATH)
if list(eval_checkpoint["class_names"]) != list(KEPT_NAMES):
    raise RuntimeError("best_model.pth 的类别顺序与当前 dataset.yaml 不一致")

y_true = []
y_pred = []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        logits = eval_model(images)
        prediction = logits.argmax(dim=1).cpu().tolist()
        y_pred.extend(prediction)
        y_true.extend(labels.tolist())

report = classification_report(
    y_true,
    y_pred,
    labels=list(range(len(KEPT_NAMES))),
    target_names=KEPT_NAMES,
    digits=4,
    zero_division=0,
)
matrix = confusion_matrix(y_true, y_pred, labels=list(range(len(KEPT_NAMES))))
correct = sum(int(left == right) for left, right in zip(y_true, y_pred))
accuracy = correct / max(len(y_true), 1)
print(f"测试准确率: {accuracy:.4f}  ({correct}/{len(y_true)})")
print(report)

eval_dir = WORKING / "resnet_eval"
eval_dir.mkdir(parents=True, exist_ok=True)
report_path = eval_dir / "classification_report.txt"
mapping = "\n".join(
    f"{index:02d}\t{name}\t{meaning}" for index, (name, meaning) in enumerate(zip(KEPT_NAMES, KEPT_MEANINGS))
)
report_path.write_text(mapping + "\n\n" + report, encoding="utf-8", newline="\n")

figure_size = max(12, len(KEPT_NAMES) * 0.35)
plt.figure(figsize=(figure_size, figure_size * 0.85))
sns.heatmap(
    matrix,
    cmap="Blues",
    xticklabels=KEPT_NAMES,
    yticklabels=KEPT_NAMES,
    square=True,
    cbar=True,
)
plt.xlabel("predicted")
plt.ylabel("true")
plt.title(f"ResNet-18 confusion matrix  acc={accuracy:.4f}")
plt.xticks(rotation=90, fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
figure_path = eval_dir / "confusion_matrix.png"
plt.savefig(figure_path, dpi=140)
plt.show()
print("报告:", report_path)
print("混淆矩阵:", figure_path)


## Cell 9 — 训练 YOLOv8

Ultralytics 在 `device=[0, 1]` 时走自己的 DDP，不能再套一层 `nn.DataParallel`，否则训练接口不可用。ResNet 已按要求使用 DataParallel。检测关闭左右翻转。


In [ ]:
from ultralytics import YOLO

if torch is None:
    import torch

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
if gpu_count >= 2:
    train_device = [0, 1]
elif gpu_count == 1:
    train_device = 0
else:
    train_device = "cpu"
print("YOLO device:", train_device, "workers: 0")

yolo = YOLO("yolov8s.pt")
yolo.train(
    data=str(DATASET_YAML),
    epochs=30,
    imgsz=640,
    batch=16,
    workers=0,
    device=train_device,
    project=str(WORKING / "yolo_runs"),
    name="tt100k45",
    amp=gpu_count > 0,
    exist_ok=True,
    fliplr=0.0,
    flipud=0.0,
    patience=30,
    seed=SEED,
)

candidates = []
trainer = getattr(yolo, "trainer", None)
if trainer is not None and getattr(trainer, "save_dir", None):
    candidates.append(Path(trainer.save_dir) / "weights" / "best.pt")
run_root = WORKING / "yolo_runs"
if run_root.exists():
    found = sorted(run_root.rglob("best.pt"), key=lambda path: path.stat().st_mtime, reverse=True)
    candidates.extend(found)
best_path = next((path for path in candidates if path.is_file()), None)
if best_path is None:
    raise FileNotFoundError("训练结束但没有找到 best.pt")
exported_yolo = WORKING / "best.pt"
shutil.copy2(best_path, exported_yolo)
print("YOLO 权重已复制到:", exported_yolo, f"({exported_yolo.stat().st_size / 1024 ** 2:.1f} MB)")


## Cell 10 — 下载权重到本地


In [ ]:
exports = [
    WORKING / "best.pt",
    WORKING / "best_model.pth",
    WORKING / "dataset.yaml",
]
print("请从 Kaggle 右侧 Output 下载下面三个文件。它们都在 /kaggle/working/。")
print()
for path in exports:
    if path.is_file():
        print(f"[就绪] {path}  ({path.stat().st_size / 1024:.1f} KB)")
    else:
        print(f"[缺失] {path}")
print(
    """
下载后放到本地项目的 weights 目录，文件名保持不变：

    weights/best.pt
    weights/best_model.pth
    weights/dataset.yaml

然后在项目根目录执行：

    pip install -r requirements.txt
    python demo.py --yolo weights/best.pt --resnet weights/best_model.pth --data weights/dataset.yaml

浏览器打开 http://127.0.0.1:8000 ，上传街景图。服务只监听本机，不会挂到局域网。
服务先用 YOLO 检出路标框，再把每个小图送进这份 ResNet，返回含义、位置和画框后的图片。
"""
)
